# Market Structured Bars

## Problem Definition

**Question.** Do the local AAPL 2025 structured-bar Parquets form valid, observed OHLCV summaries of the tick source?

**Role in the workflow.** Validate the standard, imbalance, and run-bar feature boundary and select dollar bars for downstream events.

**Inputs.** The local tick Parquet and nine local structured-bar Parquets.

**Outputs.** A schema/coverage manifest and the selected dollar-bar path; existing feature files are not regenerated.

**Why this method.** Validating stored preprocessing products keeps the clean run offline and avoids unnecessary multi-million-row recomputation.

**Assumptions.** Every bar has ordered start/end times, positive prices, nonnegative volume, and AAPL identity.

**Handoff.** The dollar-bar path to fractional differentiation, technical features, and event labeling.


## Real Data and Preprocessing Check

These files are observed preprocessing products, not simulated bars. The holdout applies to labeled events later; bar validation therefore does not inspect any model target.


In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve().parents[1]
raw_path = PROJECT_ROOT / "data/research_data/market/data/aapl_2025-01-01_2025-12-31.parquet"
feature_dir = PROJECT_ROOT / "data/research_data/market/features"
period = "2025-01-01_2025-12-31"
bar_names = [
    "tick_bar", "volume_bar", "dollar_bar",
    "tick_imbalance_bar", "volume_imbalance_bar", "dollar_imbalance_bar",
    "tick_run_bar", "volume_run_bar", "dollar_run_bar",
]

raw = pd.read_parquet(raw_path, columns=["timestamp", "symbol", "price", "size"])
raw["timestamp"] = pd.to_datetime(raw["timestamp"], utc=True)
required_bar_columns = {
    "start", "end", "symbol", "open", "high", "low", "close",
    "volume", "dollar_value", "ticks", "buy_volume", "sell_volume",
}

rows = []
for bar_name in bar_names:
    path = feature_dir / f"aapl_{bar_name}_{period}.parquet"
    bars = pd.read_parquet(path)
    assert required_bar_columns.issubset(bars.columns)
    bars["start"] = pd.to_datetime(bars["start"], utc=True)
    bars["end"] = pd.to_datetime(bars["end"], utc=True)
    assert bars["symbol"].eq("AAPL").all()
    assert bars["end"].ge(bars["start"]).all()
    assert bars[["open", "high", "low", "close"]].gt(0).all().all()
    assert bars[["volume", "dollar_value", "ticks"]].ge(0).all().all()
    rows.append(
        {
            "bar_type": bar_name,
            "rows": len(bars),
            "start_utc": bars["start"].min(),
            "end_utc": bars["end"].max(),
            "median_ticks": float(bars["ticks"].median()),
        }
    )

bar_manifest = pd.DataFrame(rows).set_index("bar_type")
dollar_bar_path = feature_dir / f"aapl_dollar_bar_{period}.parquet"
dollar_bars = pd.read_parquet(dollar_bar_path)

assert bar_manifest["start_utc"].min() >= raw["timestamp"].min()
assert bar_manifest["end_utc"].max() <= raw["timestamp"].max()
display(bar_manifest)
display(dollar_bars.head())


## Results, Limitations, and Handoff

Different thresholds produce materially different sample counts, so dollar bars are a documented workflow choice rather than proof of optimality. All validated bar files remain reproducible local inputs.

The next notebook receives the AAPL dollar-bar Parquet. No conclusion in this notebook is evidence of live-trading profitability.
